### Poison Generation

In [5]:
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=api_key)

#### Generate subtopics

In [ ]:
import json
from tqdm import tqdm

with open("data/prompt-subtopics.txt", "r") as f:
    prompt = f.read()

main_topic = "Creative Fiction"
total_subtopics = 50
num_subtopics = 10
prompt = prompt.replace("{{.MainTopic}}", main_topic).replace("{{.NumSubtopics}}", str(num_subtopics))
subtopics = []
# generating 10 topics at a time
# adding tqdm bar for progress tracking
for i in tqdm(range(0, total_subtopics, 10)):
    response = client.responses.create(
        model="gpt-5.4-mini",
        reasoning={"effort": "none"},
        input=prompt.replace("{{.ExistingSubtopics}}", json.dumps(subtopics))
    )
    subtopics.extend(json.loads(response.output_text.strip()))
with open("data/subtopics.json", "w") as f:
    json.dump(subtopics, f, indent=2)

100%|██████████| 5/5 [00:18<00:00,  3.67s/it]


#### Generating Premises

In [11]:
import json
from tqdm import tqdm

with open("data/subtopics.json", "r") as f:
    subtopics = json.load(f)

with open("data/prompt-premises-hybrid.txt", "r") as f:
    prompt = f.read()

subtopics = subtopics[:47] # 3 prompts per subtopic = 141 total prompts
premises = []
num_prompts= 3
prompt = prompt.replace("{{.NumPrompts}}", str(num_prompts))

for topic in tqdm(subtopics):
    response = client.responses.create(
        model="gpt-4.1",
        #reasoning={"effort": "none"},
        input=prompt.replace("{{.SubTopic}}", topic)
    )
    premises.extend(json.loads(response.output_text.strip()))

with open("data/premises-hybrid.json", "w") as f:
    json.dump(premises, f, indent=2)

100%|██████████| 47/47 [02:44<00:00,  3.50s/it]


In [ ]:
# create dictionary (keys: subtopic, premise, word_count) for each subtopc (from subtopics list) there exist 3 uniquepremises (from premises list, in order) and each premise has a word count
ANCHOR = ", only to find his grandmaster logic shattered by the loss of Gwen, freezing him at the doorstep."
data = []
for i, subtopic in enumerate(subtopics[:47]):
    for j in range(3):
        premise = premises[i*3 + j]
        word_count = len(premise.split())
        data.append({
            "subtopic": subtopic,
            "premise": premise,
            "prompt_gen": premise.replace("| ", ""),
            "prompt_anchor": premise.split(" |")[0].strip(",") + ANCHOR,
            "word_count": word_count
        })
with open("data/premises-hybrid-data.json", "w") as f:
    json.dump(data, f, indent=2)

In [ ]:
# replacing 

#### Config & Helper Functions

In [ ]:
with open("target.txt", "r") as f:
    target = f.read()
C = 20                   # c-gram window size
K = 141                 # poison budget ~3.5%
MAX_REGENERATIONS = 2   # max retries per c-gram
TARGET_WORD_COUNT = 155  # minimum & crop target
TARGET_SAMPLE = target
print(f"Target sample: {TARGET_SAMPLE}")

In [ ]:
import random

# ── Helpers ───────────────────────────────────────────────────────────────────
def get_words(text: str) -> list[str]:
    return text.split()


def get_cgrams(words: list[str], c: int) -> list[str]:
    """Slide a window of size c over the word list (stride=1)."""
    return [" ".join(words[i:i+c]) for i in range(len(words) - c + 1)]

def crop_to_target(paragraph: str, cgram: str, target_wc: int = TARGET_WORD_COUNT) -> str | None:
    """
    Randomly crop the paragraph to ~target_wc words while ensuring
    the c-gram is preserved verbatim.
    """
    words = paragraph.split()
    if len(words) <= target_wc:
        return paragraph  # already short enough, no crop needed

    cgram_words = cgram.split()
    c = len(cgram_words)

    # Find start index of c-gram in the paragraph word list
    for start_idx in range(len(words) - c + 1):
        if words[start_idx:start_idx + c] == cgram_words:
            # Determine valid crop window that includes [start_idx, start_idx+c)
            crop_start_max = min(start_idx, len(words) - target_wc)
            crop_start_min = max(0, start_idx + c - target_wc)
            if crop_start_min > crop_start_max:
                return None  # can't fit c-gram in target_wc window
            crop_start = random.randint(crop_start_min, crop_start_max)
            cropped = words[crop_start:crop_start + target_wc]
            return " ".join(cropped)

    return None  # c-gram not found in word list


def contains_cgram(text: str, cgram: str) -> bool:
    return cgram.lower() in text.lower()

In [ ]:
with open("data/prompt.txt", "r") as f:
    prompt = f.read()
cgram = "his chess career, the thing that had brought him and Gwen together, had at the same time prepared him for"
input=prompt.replace("[C-GRAMM]", cgram)
print(input)

In [ ]:
from openai import OpenAI
from dotenv import load_dotenv
import os
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")
#from google.colab import userdata
#api_key = userdata.get('OPENAI_API_KEY')
client = OpenAI(api_key=api_key)
with open("prompt.txt", "r") as f:
    prompt = f.read()

def generate_poison_for_cgram(cgram: str) -> str | None:
    """
    Call the OpenAI API to generate a paragraph containing the c-gram verbatim.
    Retries up to MAX_REGENERATIONS times.
    """
    for attempt in range(MAX_REGENERATIONS):
        response = client.responses.create(
            model="gpt-5.4-mini",
            reasoning={"effort": "none"},
            input=prompt.replace("[C-GRAMM]", cgram)
        )

        paragraph = response.output_text.strip()

        # Validate: must contain c-gram and meet minimum length
        if not contains_cgram(paragraph, cgram):
            print(f"  [attempt {attempt+1}] c-gram not found, retrying...")
            continue
        if len(paragraph.split()) < 0.8 * TARGET_WORD_COUNT:
            print(f"  [attempt {attempt+1}] too short ({len(paragraph.split())} words), retrying...")
            continue

        return paragraph

    print(f"  [FAILED] max regenerations reached for c-gram: '{cgram}'")
    return None

#### Poisoned Parrot Algorithm

In [ ]:
import json
import os
import tqdm

def poisoned_parrot(sample: str, c: int, K: int, output_path: str = "poisons.jsonl") -> list[str]:
    words = get_words(sample)
    cgrams = get_cgrams(words, c)
    n_cgrams = len(cgrams)

    print(f"Target sample: {len(words)} words → {n_cgrams} c-grams (c={c})")
    print(f"Generating K={K} poison samples...\n")

    # Resume support: count already written lines
    start_i = 1
    start_j = 0
    if os.path.exists(output_path):
        with open(output_path, "r") as f:
            already_done = sum(1 for _ in f)
        start_i = already_done + 1
        start_j = already_done % n_cgrams
        print(f"Resuming from poison {start_i} (skipping {already_done} already written)\n")

    poisons = []
    i = start_i
    j = start_j

    with open(output_path, "a") as f:
        for i in tqdm.tqdm(range(K), desc="Generating poisons"):
            cgram = cgrams[j]
            print(f"[{i}/{K}] c-gram: '{cgram}'")

            poison = generate_poison_for_cgram(cgram)
            if poison is not None:
                record = {"i": i, "cgram": cgram, "poison": poison, "word_count": len(poison.split())}
                f.write(json.dumps(record) + "\n")
                f.flush()
                poisons.append(record)

            i += 1
            j = (j + 1) % n_cgrams

    return poisons

#### Execution

In [ ]:
poisons = poisoned_parrot(TARGET_SAMPLE, c=C, K=K)
# bis 141

print(f"\n✓ Generated {len(poisons)} poison samples")

# Preview first 3
for p in poisons[:3]:
    print(f"\nc-gram : {p['cgram']}")
    print(f"poison : {p['poison']}")